### Revision Analysis Using TRACE

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd


In [2]:
from trace_revision import trace, make_newlines_visible

In [3]:
INPUT_XLSX = Path("../data/all_data_asurite_with_uuid.xlsx")
OUTPUT_DIR = Path("../output")

UUID_COL = "uuid"
DRAFT_COL = "essay_text"
REVISED_COL = "language_revision_text"

In [4]:
REPLACE_SIM_THRESHOLD = 0.4

# Full Step3 schema we implemented (keep this order in outputs)
FINAL_COLS = [
    "operation_id", "operation", "chunk_label",
    "source_chunk", "source_chunk_index", "source", "source_index",
    "target_chunk", "target_chunk_index", "target", "target_index",
]

In [5]:
def sanitize_uuid(u: Any) -> str:
    """Make a safe filename stem from uuid-like values."""
    s = "" if u is None else str(u).strip()
    if not s:
        return "missing_uuid"
    s = re.sub(r"[^A-Za-z0-9_\-]+", "_", s)
    return s[:200]


def normalize_text(x: Any) -> str:
    """Convert cell values to text; treat NaN/None as empty string."""
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    return str(x)


def ensure_cols(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    """Ensure required columns exist, then return projected df in the given order."""
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]


def project_dict_rows(rows: List[Dict[str, Any]], cols: List[str]) -> List[Dict[str, Any]]:
    """Project list-of-dict rows to only the specified cols (stable order)."""
    return [{c: r.get(c, None) for c in cols} for r in rows]


def run_one(draft: str, revised: str) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    step3_df, step3_dict, _dbg = trace(
        draft,
        revised,
        replace_similarity_threshold=REPLACE_SIM_THRESHOLD,
        debug=False,
    )
    out_df = ensure_cols(step3_df, FINAL_COLS)
    out_df = make_newlines_visible(out_df)
    out_dict = project_dict_rows(step3_dict, FINAL_COLS)
    return out_df, out_dict


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
df_in = pd.read_excel(INPUT_XLSX)

In [8]:
failures: List[Dict[str, Any]] = []
total = len(df_in)

In [9]:
for idx, row in df_in.iterrows():
        uuid_raw = row.get(UUID_COL)
        uuid_safe = sanitize_uuid(uuid_raw)

        draft = normalize_text(row.get(DRAFT_COL))
        revised = normalize_text(row.get(REVISED_COL))

        csv_path = OUTPUT_DIR / f"{uuid_safe}.csv"
        json_path = OUTPUT_DIR / f"{uuid_safe}.txt"

        if not draft.strip() and not revised.strip():
            failures.append(
                {"row_index": int(idx), "uuid": str(uuid_raw), "error": "Both draft and revised texts are empty."}
            )
            continue

        try:
            out_df, out_dict = run_one(draft, revised)

            # CSV with all columns (in FINAL_COLS order)
            out_df.to_csv(csv_path, index=False, encoding="utf-8")

            # JSON string (.txt) with the same columns/order
            json_str = json.dumps(out_dict, ensure_ascii=False)
            json_path.write_text(json_str, encoding="utf-8")

        except Exception as e:
            failures.append({"row_index": int(idx), "uuid": str(uuid_raw), "error": repr(e)})

        if (idx + 1) % 25 == 0 or (idx + 1) == total:
            print(f"Processed {idx + 1}/{total}")


Processed 25/113
Processed 50/113
Processed 75/113
Processed 100/113
Processed 113/113


In [10]:

if failures:
    fail_path = OUTPUT_DIR / "_trace_failures.csv"
    pd.DataFrame(failures).to_csv(fail_path, index=False, encoding="utf-8")
    print(f"Done with {len(failures)} failures. See: {fail_path}")
else:
    print("Done. No failures.")

Done. No failures.


### Process files in the data folder using the pipeline 

In [2]:
data_path = "../data/"

In [2]:
from trace_revision import trace, make_newlines_visible

In [3]:
from trace_revision.general import get_files_in_directory

In [4]:
all_files = get_files_in_directory(data_path)
print(f"Found {len(all_files)} files in {data_path}")

Found 110 files in ../data/


In [11]:
from typing import Any, Dict, List, Optional, Tuple
import pandas as pd

def run_trace(draft: str, revised: str) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    out_df, out_dict, _dbg = trace(
        draft,
        revised,
        replace_similarity_threshold=0.4,
        debug=False,
    )

    out_df = make_newlines_visible(out_df)
    return out_df

In [12]:
from pathlib import PureWindowsPath

In [13]:
for afile in all_files:
    print(f"Processing: {afile}")
    try:
        df = pd.read_csv(afile)
        draft = df["source_text"].iloc[0]
        revised = df["revised_text"].iloc[0]
        out_df = run_trace(draft, revised)
        # save out_df to a new csv in an output directory with the same filename
        df_basename = PureWindowsPath(afile).name
        out_df.to_csv(f"../output/{df_basename}", index=False, encoding="utf-8")
    except Exception as e:
        print(f"Error processing {afile}: {e}")


Processing: ../data/046209b6-0bdb-42af-95db-eb63a8475548_0acfcc03-567d-4d53-8656-420e4b167e29_trace.csv
Processing: ../data/046209b6-0bdb-42af-95db-eb63a8475548_253ba75c-5977-477b-8d74-c55ebede425d_trace.csv
Processing: ../data/046209b6-0bdb-42af-95db-eb63a8475548_8bf8f41b-eb1b-478a-9331-8d0ad56a42dc_trace.csv
Processing: ../data/046209b6-0bdb-42af-95db-eb63a8475548_9e81803a-06e7-4553-920a-0a970238a3bf_trace.csv
Processing: ../data/046209b6-0bdb-42af-95db-eb63a8475548_bdfb9676-e977-4ab3-bcdd-6de37654dd97_trace.csv
Processing: ../data/14a8cc50-42a4-4d62-8cf3-11c44d9203fa_02459af4-b0f7-49a7-a038-b270d1e49742_trace.csv
Processing: ../data/14a8cc50-42a4-4d62-8cf3-11c44d9203fa_133a54e6-cef9-42c6-9153-4df0cbe8cb6c_trace.csv
Processing: ../data/14a8cc50-42a4-4d62-8cf3-11c44d9203fa_22fb5ddb-bad2-4427-aee1-292ef77295b2_trace.csv
Processing: ../data/14a8cc50-42a4-4d62-8cf3-11c44d9203fa_28437c6c-81ba-443e-a496-65cfa7b2cf1c_trace.csv
Processing: ../data/14a8cc50-42a4-4d62-8cf3-11c44d9203fa_ed131fc

## Process Data in Batches from a Single CSV or Excel File

In [9]:
data_dir = "../data/essays_andrew.xlsx"

In [10]:
from trace_revision import trace, make_newlines_visible

In [11]:
from typing import Any, Dict, List, Optional, Tuple
import pandas as pd

def run_trace(draft: str, revised: str) -> Tuple[pd.DataFrame, List[Dict[str, Any]]]:
    out_df, out_dict, _dbg = trace(
        draft,
        revised,
        replace_similarity_threshold=0.4,
        debug=False,
    )

    out_df = make_newlines_visible(out_df)
    return out_df

In [12]:
import pandas as pd

In [13]:
data_df = pd.read_excel(data_dir)

In [15]:
data_df.columns

Index(['asurite', 'condition', 'Draft_1', 'Draft_2'], dtype='str')

In [16]:
from pathlib import Path
output_dir = Path("../output/andrew")

In [17]:

# process each row in data_df 
for index, row in data_df.iterrows():
    out_df = run_trace(row["Draft_1"], row["Draft_2"])

    file_name = f"{row['asurite']}_{row['condition']}.xlsx"
    out_df.to_excel(output_dir / file_name, index = False)
